In [7]:
# Improtar librerias
import pandas as pd
import glob
import os
import re

In [8]:
import os
import glob
import re
import pandas as pd

ruta_base = "../data/fbref/"
ruta_procesada = os.path.join(ruta_base, "processed")
os.makedirs(ruta_procesada, exist_ok=True)

# Configuro dónde buscar y como leer cada tipo de tabla
config_archivos = {
    "partidos": {"ruta": "partidos/*.csv", "header": 0},
    "tabla_general": {"ruta": "tabla general/*.csv", "header": 0},
    "equipos": {"ruta": "equipos/*.csv", "header": 1},
    "jugadores": {"ruta": "jugadores/*.csv", "header": 1}
}

# Función para extraer el año del nombre del archivo (ej. "partidos2026.csv" -> "2026")
def extraer_temporada(nombre_archivo):
    numeros = re.search(r"(\d+)", os.path.basename(nombre_archivo))
    if numeros:
        anio = numeros.group(1)
        # Si el año viene como "26", lo convertimos a "2026"
        if len(anio) == 2:
            return f"20{anio}"
        return anio[:4]
    return "Desconocido"

for categoria, config in config_archivos.items():
    ruta_busqueda = os.path.join(ruta_base, config["ruta"])
    archivos_csv = glob.glob(ruta_busqueda)
    
    if not archivos_csv:
        print(f"No se encontraron archivos para: {categoria}")
        continue
        
    print(f"\nProcesando {categoria} ({len(archivos_csv)} archivos)...")
    
    df_year = []
    
    for archivo in archivos_csv:
        try:
            # Leemos respetando si FBref metió doble encabezado o no
            df = pd.read_csv(archivo, header=config["header"])
            
            # Quito columnas vacías o "Unnamed" que genera FBref
            df = df.loc[:, ~df.columns.str.contains("^Unnamed")]
            
            # Si es partidos, quitamos las filas de semanas, no es de utilidad para el análisis y puede generar errores al concatenar
            if categoria == "partidos":
                df = df.dropna(subset=["Wk", "Home", "Away"])
                df = df[df["Wk"] != "Wk"]
            
            # Meto el año para saber de cuándo es cada dato en el csv
            temporada = extraer_temporada(archivo)
            df.insert(0, "Temporada", temporada)
            
            df_year.append(df)
            
        except Exception as e:
            print(f"Error al leer {os.path.basename(archivo)}: {e}")
            
    # Unimos todos los años verticalmente
    if df_year:
        df_general = pd.concat(df_year, ignore_index=True)
        
        # Exportamos el CSV general de esa categoría
        ruta_salida = os.path.join(ruta_procesada, f"{categoria}_historico_completo.csv")
        df_general.to_csv(ruta_salida, index=False, encoding="utf-8-sig")
        print(f"{categoria.upper()} lista: {len(df_general):,} filas exportadas a {ruta_salida}")

print("\nFinalizado el procesamiento de todas las categorías.")


Procesando partidos (21 archivos)...
PARTIDOS lista: 7,980 filas exportadas a ../data/fbref/processed\partidos_historico_completo.csv

Procesando tabla_general (21 archivos)...
TABLA_GENERAL lista: 421 filas exportadas a ../data/fbref/processed\tabla_general_historico_completo.csv

Procesando equipos (21 archivos)...
EQUIPOS lista: 420 filas exportadas a ../data/fbref/processed\equipos_historico_completo.csv

Procesando jugadores (21 archivos)...
JUGADORES lista: 11,765 filas exportadas a ../data/fbref/processed\jugadores_historico_completo.csv

Finalizado el procesamiento de todas las categorías.
